In [1]:
import pandas as pd
df = pd.read_csv("multiple_linear_regression_dataset.csv")

In [2]:
df.head()

,age,experience,income
0,25,1,30450
1,30,3,35670
2,47,2,31580
3,32,5,40130
4,43,10,47830


In [3]:
df.columns

Index(['age', 'experience', 'income'], dtype='object')

In [4]:
df.shape

(20, 3)

In [5]:
# Which columns serve as inputs to the model?
# age and experience

# Which column is the target we want to predict?
# income

# How many input features does the model need to handle?
# 2

In [6]:
X = df[["age", "experience"]].values
y = df["income"].values

In [7]:
# What is the shape of X?
# (20, 2)

# What is the shape of y?
# (20,)

# Why does X have 2 columns while y has only one?
# X holds two input features (age and experience) that the model uses for prediction, whereas y holds the single continuous target variable (income) that we are trying to estimate.

In [8]:
import numpy as np
n_features = X.shape[1]
weights = np.zeros(n_features)
bias = 0.0

In [9]:
# Why do we need one weight per feature?
# Each feature contributes differently to the predicted output, so a separate weight is needed to capture the individual influence of age and experience on income.

# Why is the bias kept separate?
# The bias acts as an intercept, allowing the model to produce non-zero predictions even when all input features equal zero. It shifts the entire regression line, enabling a better overall fit.

# Is it risky to initialize weights with large values?
# Yes. Large initial weights can produce enormous gradient updates that cause the training process to overshoot the optimal solution repeatedly, potentially preventing convergence altogether.

In [10]:
def predict(X, weights, bias):
    y_hat = np.dot(X, weights) + bias
    return y_hat

In [11]:
# Why is there no activation function here?
# Linear regression models a continuous target, so the raw linear combination is the final output — no transformation is needed.

# What range of values can y_hat take?
# Any real number, which is exactly what we need when predicting a continuous variable like income.

# How does this differ from logistic regression?
# Logistic regression targets class probabilities bounded between 0 and 1, so it passes the linear score through a sigmoid activation. Linear regression omits this step because an unbounded output is appropriate for regression.

In [12]:
def mean_squared_error(y_true, y_pred):
    mse = np.mean((y_true - y_pred) ** 2)
    return mse

In [13]:
# Why do we square the errors?
# Squaring makes all errors positive and amplifies larger mistakes, encouraging the optimizer to reduce significant deviations more aggressively.

# What effect does a single very wrong prediction have?
# A large prediction error squares to a very large value, which can dominate the mean squared error and pull the gradient updates strongly in its direction. This sensitivity to outliers is a well-known property of MSE.

# Why not use the absolute value instead?
# Mean absolute error treats all errors proportionally without extra penalty for large mistakes. MSE is preferred when penalizing large errors more heavily is desirable, though it makes the model less robust to outliers compared to MAE.

In [14]:
def compute_gradients(X, y_true, y_pred):
    n = len(y_true)
    dw = (-2 / n) * np.dot(X.T, (y_true - y_pred))
    db = (-2 / n) * np.sum(y_true - y_pred)
    return dw, db

In [15]:
# Why does X appear in dw but not in db?
# The weight gradient reflects how much each input feature contributed to the error, so X must be included. The bias gradient only depends on the total residual, not on any specific input, so X is absent from db.

# Why does the error term appear in both gradient expressions?
# The error quantifies how far predictions deviate from ground truth. Both the weight and bias updates are derived from the chain rule of the loss with respect to these parameters, and the error is the common factor in both derivative paths.

# What happens when the error is exactly zero?
# Both dw and db become zero, so no parameter update occurs. The model has found a perfect fit on the training data and gradient descent has effectively converged.

In [16]:
def update_parameters(weights, bias, dw, db, lr):
    weights = weights - lr * dw
    bias = bias - lr * db
    return weights, bias

In [17]:
lr = 0.0001
epochs = 1000

for epoch in range(epochs):
    y_hat = predict(X, weights, bias)
    current_loss = mean_squared_error(y, y_hat)
    dw, db = compute_gradients(X, y, y_hat)
    weights, bias = update_parameters(weights, bias, dw, db, lr)
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {current_loss}")

Epoch 0, Loss: 1727049635.0
Epoch 100, Loss: 66491868.553113505
Epoch 200, Loss: 61752567.2011901
Epoch 300, Loss: 58616531.07847047
Epoch 400, Loss: 56528801.53951118
Epoch 500, Loss: 55126542.02946697
Epoch 600, Loss: 54172526.94885705
Epoch 700, Loss: 53511656.14292053
Epoch 800, Loss: 53042523.72795741
Epoch 900, Loss: 52698829.56325033


In [18]:
# Does the loss decrease over time?
# Yes. With each epoch the model refines its parameters based on the gradient signal, progressively closing the gap between predictions and true values and reducing MSE.

# What does it mean if the loss increases instead?
# An increasing loss usually signals that the learning rate is too high. Excessively large updates cause the parameters to overshoot the minimum, and the model diverges rather than converging.

# How do the learning rate and number of epochs interact?
# The learning rate controls the magnitude of each parameter update, while epochs determine how many updates are applied. A large learning rate can reach the vicinity of the optimum faster but risks instability; a small learning rate is more stable but needs many more epochs to converge. The two must be balanced to achieve efficient, stable training.

In [19]:
print(f"Final weights: {weights}")
print(f"Final bias: {bias}")

new_candidate = np.array([[30, 5]])
predicted_income = predict(new_candidate, weights, bias)
print(f"Predicted income: {predicted_income[0]}")

Final weights: [ 764.75405919 1371.03430441]
Final bias: 321.7364117447249
Predicted income: 30119.529709500806


In [20]:
# Is the prediction reasonable?
# Whether the prediction is sensible depends on the income range observed in the training data. If the value falls within a plausible interval given the input features, it can be considered reasonable. Without full context of the dataset distribution, a definitive judgment is difficult.

# Does the model interpolate smoothly?
# Yes. Because linear regression defines a hyperplane over the feature space, predictions for any new input are a smooth linear function of the learned weights and bias — no abrupt jumps occur between training points.

# Why is this approach better than threshold-based rules?
# Threshold rules produce discrete, step-like outputs that cannot express degrees of difference. A linear regression model outputs a continuous value that naturally reflects varying levels of the input features, making it more expressive and accurate for problems where the relationship is graded rather than categorical.